In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks

In [ ]:
FOLDER    = "mechanical test/"
N_CYCLES  = 5
STRAIN    = 0.5
MA_WINDOW = 50

sensors = {
    "50k test 1": {"file": "50kohm_1_test_adc_data_20260612_123857.csv", "R0": 50e3 },
    "50k test 2": {"file": "50kohm_2_test_adc_data_20260612_124824.csv", "R0": 50e3 },
    "50k test 3": {"file": "50kohm_3_test_adc_data_20260612_125644.csv", "R0": 50e3 },
    "150k":       {"file": "150kohm-mechanical.csv",                      "R0": 150e3},
    "200k":       {"file": "200kohm_test_adc_data_20260612_130535.csv",   "R0": 200e3},
    "210k":       {"file": "210kohm-mechanical.csv",                      "R0": 210e3},
    "250k":       {"file": "250kohm_test_adc_data_20260612_123025.csv",   "R0": 250e3},
    "460k":       {"file": "460kohm-mechanical.csv",                      "R0": 460e3},
}

# Cut data after this many SECONDS — data after this point is discarded
CUTOFF_S = {
    "460k": 170,
}

dfs = {}
for label, meta in sensors.items():
    df = pd.read_csv(FOLDER + meta["file"], parse_dates=["Timestamp"])
    df["Channel_1"] = (
        df["Channel_1"]
        .replace(-1.0, np.nan)
        .interpolate(method="linear")
        .bfill()
    )
    df["elapsed_s"] = (df["Timestamp"] - df["Timestamp"].iloc[0]).dt.total_seconds()
    if label in CUTOFF_S:
        cutoff   = CUTOFF_S[label]
        n_before = len(df)
        df       = df[df["elapsed_s"] <= cutoff].reset_index(drop=True)
        print(f"{label}: cut at {cutoff}s -> kept {len(df)}/{n_before} samples, max t={df['elapsed_s'].iloc[-1]:.1f}s")
    df["Channel_1_ma"] = df["Channel_1"].rolling(window=MA_WINDOW, center=True, min_periods=1).mean()
    dfs[label] = df
    print(f"{label}: {len(df)} samples, {df['elapsed_s'].iloc[-1]:.1f}s")

## Cycle detection

**Pipeline:**
1. Apply **moving average** (`MA_WINDOW` samples) to reduce noise
2. Clip signal to p99 to remove ADC saturation spikes
3. Find peaks with **minimum inter-peak distance** = 60% of one expected cycle duration
4. **R0 per cycle** = minimum of filtered signal between previous peak and current peak
5. `GF = ((R_max - R0) / R0) / 0.5`

In [ ]:
def find_cycle_peaks(arr, n_cycles=5):
    sig_range = arr.max() - arr.min()
    min_dist  = max(1, int(0.6 * len(arr) / n_cycles))
    for frac in [0.25, 0.15, 0.05, 0.0]:
        peaks, props = find_peaks(arr, prominence=sig_range * frac, distance=min_dist)
        if len(peaks) >= n_cycles:
            break
    if len(peaks) > n_cycles:
        prom  = props.get("prominences", arr[peaks])
        peaks = np.sort(peaks[np.argsort(prom)[-n_cycles:]])
    return peaks


all_results = {}

for label, meta in sensors.items():
    df      = dfs[label]
    sig_ma  = df["Channel_1_ma"].to_numpy(dtype=float)
    t       = df["elapsed_s"].to_numpy()
    p99     = np.percentile(sig_ma, 99)
    clipped = np.clip(sig_ma, 0, p99)
    peaks   = find_cycle_peaks(clipped, N_CYCLES)
    cycles  = []
    for i, pidx in enumerate(peaks):
        R_max     = clipped[pidx]
        seg_start = int(peaks[i - 1]) if i > 0 else 0
        segment   = clipped[seg_start:pidx]
        if len(segment) > 0:
            v_idx = seg_start + int(np.argmin(segment))
            R0    = clipped[v_idx]
        else:
            v_idx = seg_start
            R0    = meta["R0"]
        GF = ((R_max - R0) / R0) / STRAIN
        cycles.append({"cycle": i+1, "t_peak": t[int(pidx)], "t_valley": t[v_idx],
                       "R_max": R_max, "R0": R0, "GF": GF})
    all_results[label] = {"cycles": cycles, "p99": p99}
    gf_vals = [c["GF"] for c in cycles]
    print(f"{label:12s}  peaks={len(peaks)}  GF={[f'{g:.3f}' for g in gf_vals]}  mean={np.nanmean(gf_vals):.3f}")

## Cycles plot

In [ ]:
colors = ["#1f77b4","#ff7f0e","#2ca02c","#8c564b","#d62728","#e377c2","#9467bd","#17becf"]
fig, axes = plt.subplots(len(sensors), 1, figsize=(14, 4.0*len(sensors)), sharex=False)
for ax, (label, df), color in zip(axes, dfs.items(), colors):
    sig_raw = df["Channel_1"].to_numpy(dtype=float)
    sig_ma  = df["Channel_1_ma"].to_numpy(dtype=float)
    t       = df["elapsed_s"].to_numpy()
    p99     = all_results[label]["p99"]
    cycles  = all_results[label]["cycles"]
    ax.plot(t, np.clip(sig_raw, 0, p99), color=color, lw=0.5, alpha=0.25, label="Raw")
    ax.plot(t, np.clip(sig_ma,  0, p99), color=color, lw=1.5, alpha=0.95, label=f"MA ({MA_WINDOW} samples)")
    for c in cycles:
        ax.plot(c["t_peak"],   c["R_max"], "v", color="red",   ms=9, zorder=4, label="R_max" if c["cycle"]==1 else "")
        ax.plot(c["t_valley"], c["R0"],    "^", color="green", ms=9, zorder=4, label="R0"    if c["cycle"]==1 else "")
        ax.vlines(c["t_peak"],   c["R0"], c["R_max"],  color="grey",  ls="--", lw=1.2, zorder=2)
        ax.hlines(c["R0"], c["t_valley"], c["t_peak"], color="green", ls=":",  lw=1.0, zorder=2)
        txt = f"C{c['cycle']}\nRmax={c['R_max']/1e3:.1f}k\nR0  ={c['R0']/1e3:.1f}k\nGF  ={c['GF']:.3f}"
        ax.annotate(txt, xy=(c["t_peak"], (c["R0"]+c["R_max"])/2),
                    xytext=(8,0), textcoords="offset points", fontsize=7.5, va="center",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.88))
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Resistance (Ohm)")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle(f"Strain cycles  |  MA window={MA_WINDOW} samples  |  R_max red  R0 green", fontsize=12, y=1.002)
plt.tight_layout()
plt.savefig("cycles_detected.png", dpi=150, bbox_inches="tight")
plt.show()

## GF per cycle

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x       = np.arange(1, N_CYCLES + 1)
n       = len(sensors)
width   = 0.8 / n
offsets = np.linspace(-(n-1)/2, (n-1)/2, n) * width
for (label, meta), color, offset in zip(sensors.items(), colors, offsets):
    gf_vals = [c["GF"] for c in all_results[label]["cycles"]]
    while len(gf_vals) < N_CYCLES:
        gf_vals.append(np.nan)
    ax.bar(x + offset, gf_vals, width=width, color=color, edgecolor="black", linewidth=0.5, label=label, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f"Cycle {i}" for i in x])
ax.set_ylabel("Gauge Factor")
ax.set_title("Gauge Factor per cycle per sensor  (strain = 0.5)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("gf_per_cycle.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary table

In [ ]:
rows = []
for label in sensors:
    cycles  = all_results[label]["cycles"]
    row     = {"Sensor": label}
    gf_vals = []
    for c in cycles:
        row[f"GF C{c['cycle']}"] = f"{c['GF']:.3f}"
        gf_vals.append(c["GF"])
    while len(gf_vals) < N_CYCLES:
        gf_vals.append(np.nan)
        row[f"GF C{len(gf_vals)}"] = "--"
    row["Mean GF"] = f"{np.nanmean(gf_vals):.3f}"
    row["Std GF"]  = f"{np.nanstd(gf_vals):.3f}"
    rows.append(row)
pd.DataFrame(rows).set_index("Sensor")